<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 03: Foundation Model Training

We pretrain a **decoder-only foundation model** on the tokenized account sequences from
notebook 02 with **causal language modeling** (next-token prediction). No labels are
used: the training signal is the sequence itself.

```
Input:  <bos>  AMT_12  TDIF_7  DC_D  SIGN_NEG  MVC_17  CPTY_1183  ...
Target:        AMT_12  TDIF_7  DC_D  SIGN_NEG  MVC_17  CPTY_1183  BIC_17  ...
```

Every position contributes a gradient, so the model learns amount rhythms, counterparty
relationships and timing patterns per account.

---

### Three things differ from the TabFormer blueprint

1. **`vocab_size` comes from the data.** The bank tokenizer is fitted, so the embedding
   table size is only known after notebook 02. This notebook reads it from
   `corpus_report.json` and passes it as a config override. A `vocab_size` **larger**
   than the tokenizer's is harmless (unused rows); **smaller** crashes with an index
   error, or worse, silently trains on wrapped IDs if something clamps.
2. **The model is pretrained from scratch.** A checkpoint trained on another
   vocabulary (such as the blueprint's card-transaction model) cannot be reused:
   the same token ID means something entirely different here.
3. **The training loop is plain PyTorch, not NeMo AutoModel.** `src/train_decoder.py`
   replaces `TrainFinetuneRecipeForNextTokenPrediction`. At ~29M parameters on one GPU,
   FSDP2 is inapplicable and every other piece of the recipe is a one-liner: HF causal
   LMs compute masked cross-entropy from `labels` themselves, `FinancialCLMDataset` is
   already a `torch.utils.data.Dataset`, and `save_consolidated: true` is
   `save_pretrained()`. The `nemo-automodel` dependency also pulls `torchao`, whose
   float8 import (`ScalingType`) requires a newer torch than the cuDF/CuPy stack that
   notebooks 01 and 02 run on -- so keeping it would mean risking the parts that work.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.cuda")
warnings.filterwarnings("ignore", message="The '.*' attribute with value.*was provided")

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.experiment import shared_data_dir
DATA_DIR = shared_data_dir()          # set in configs/paths.yaml (the only place)
CORPUS_DIR = DATA_DIR / "decoder_corpus"
TOKENIZER_DIR = DATA_DIR / "tokenizer"

TRAIN_CORPUS = CORPUS_DIR / "train_corpus.txt"
VAL_CORPUS = CORPUS_DIR / "val_corpus.txt"
VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
REPORT_PATH = CORPUS_DIR / "corpus_report.json"

YAML_CONFIG = PROJECT_ROOT / "configs/pretrain_bank_decoder.yaml"
TRAIN_SCRIPT = PROJECT_ROOT / "scripts/train_bank_decoder.py"
MODEL_DIR = DATA_DIR / "model"        # interactive runs; experiments use their run folder
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
# Run by scripts/run_experiment.py: every path comes from the run folder.
# Interactive use (LTM_EXPERIMENT unset): EXP is None and nothing changes.
from src.experiment import active_experiment
EXP = active_experiment()
if EXP is not None:
    globals().update(EXP.paths())

for path in (TRAIN_CORPUS, VAL_CORPUS, VOCAB_PATH, REPORT_PATH):
    assert path.exists(), f"Missing {path}  (run notebook 02 first)"
assert YAML_CONFIG.exists(), f"Missing config: {YAML_CONFIG}"
assert TRAIN_SCRIPT.exists(), f"Missing training script: {TRAIN_SCRIPT}"

report = json.loads(REPORT_PATH.read_text())
VOCAB_SIZE = int(report["vocab_size"])
SEQ_LENGTH = int(report["context_window"])
TOKENS_PER_TXN = int(report["tokens_per_transaction"])
CHUNK_SIZE = int(report["chunk_size"])

n_train_lines = report["splits"]["train"]["sequences"]
approx_tokens = n_train_lines * SEQ_LENGTH

print(f"Train corpus  : {TRAIN_CORPUS}  ({n_train_lines:,} sequences)")
print(f"Val corpus    : {VAL_CORPUS}  ({report['splits']['val']['sequences']:,} sequences)")
print(f"Vocabulary    : {VOCAB_PATH}  (vocab_size={VOCAB_SIZE})")
print(f"Sequence      : {SEQ_LENGTH} tokens = {CHUNK_SIZE} txns x {TOKENS_PER_TXN} tokens + separators")
print(f"Upper bound on training tokens: ~{approx_tokens / 1e6:.0f}M")
print(f"Checkpoints   : {CHECKPOINT_DIR}")

## Model Configuration

| Parameter | Value | Notes |
|---|---|---|
| Architecture | Decoder-only Transformer (Llama config) | causal attention, RoPE, GQA, SwiGLU, RMSNorm |
| Hidden size | 512 | |
| Layers | 8 | |
| Attention heads | 8 query / 2 KV | grouped-query attention |
| FFN size | 1408 | SwiGLU |
| Context window | 4096 trained (8192 RoPE max) | ~272 transactions of history |
| Parameters | ~29M | dominated by the embedding table |
| Vocab size | from `corpus_report.json` | fitted tokenizer |

The architecture is deliberately small: with a domain vocabulary of a few thousand
tokens, model capacity is better spent on depth over sequences than on width. Swap in
any HuggingFace-compatible decoder by editing `model.config._target_` in the YAML --
whatever config class you name is instantiated with the remaining keys and handed to
`AutoModelForCausalLM.from_config`.

### Sizing guidance for your extract

The demo below runs 30 steps to prove the loss moves. For a real run, scale steps with
corpus size rather than copying a fixed number: the random-init baseline loss is
`ln(vocab_size)`, and a healthy run drops well below it within a few hundred steps.
Watch validation loss for the point where it stops improving -- with a few hundred
million tokens, that is typically a few thousand steps at this size.

In [ ]:
print(f"=== {YAML_CONFIG} ===\n")
print(YAML_CONFIG.read_text())

## 1. Launch Training

`src/train_decoder.py` holds the loop: bf16 autocast, AdamW, linear warmup into cosine
decay to `min_lr`, gradient accumulation to hold `global_batch_size` constant, gradient
clipping at 1.0, and HuggingFace-format checkpoints. The two custom components are
unchanged -- the tokenizer vocabulary (`vocab.json`) and the dataset builder
(`src/clm_data.py:build_bank_clm_dataset`), still resolved from the YAML `_target_`.

**The same `configs/pretrain_bank_decoder.yaml` remains the single source of truth.**
The loop reads `model.config`, `dataset`, `validation_dataset`, `step_scheduler`,
`optimizer`, `lr_scheduler`, `rng` and `checkpoint`; the NeMo-only sections
(`distributed`, `loss_fn`, `dataloader`, `dist_env`) are ignored. There is deliberately
no second config file -- one horizon value living in three places was enough.

Training runs **in-process** rather than as a subprocess, so the loss lands in this
cell's output instead of the kernel's terminal, and the history comes back as a dict
for plotting. (`train_bank_decoder` also takes a `progress_cb` if you want to drive a
live widget.)

The cell below runs a **30-step demo on 1 GPU**. Expect the loss to fall from about
`ln(vocab_size)` toward ~5-6.

> **If you hit OOM:** loss is computed over `local_batch_size` x `seq_length` x
> `vocab_size` logits, upcast to fp32 by cross-entropy -- ~3.3 GB at 16 x 4096 x 8232.
> Set `--step_scheduler.local_batch_size 4`; accumulation keeps the global batch at 16,
> so the optimizer math is identical.

In [ ]:
import math

from src.train_decoder import apply_overrides, load_config, train_bank_decoder

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MAX_STEPS = 30
LOCAL_BATCH_SIZE = 16   # lower to 4 if you OOM; global batch is held at 16
VAL_BATCHES = 50        # cap the validation pass; the full val set is ~1,500 batches

BASELINE_LOSS = math.log(VOCAB_SIZE)
print(f"Random-init baseline loss: ln({VOCAB_SIZE}) = {BASELINE_LOSS:.2f}\n")

# Run-specific values only; everything else comes from the YAML.
OVERRIDES = [
    "--dataset.data_path", str(TRAIN_CORPUS),
    "--dataset.vocab_path", str(VOCAB_PATH),
    "--dataset.seq_length", str(SEQ_LENGTH),
    "--validation_dataset.data_path", str(VAL_CORPUS),
    "--validation_dataset.vocab_path", str(VOCAB_PATH),
    "--validation_dataset.seq_length", str(SEQ_LENGTH),
    "--model.config.vocab_size", str(VOCAB_SIZE),
    "--step_scheduler.max_steps", str(MAX_STEPS),
    "--step_scheduler.global_batch_size", "16",
    "--step_scheduler.local_batch_size", str(LOCAL_BATCH_SIZE),
    "--lr_scheduler.lr_warmup_steps", "10",
    "--step_scheduler.val_every_steps", "15",
    "--step_scheduler.val_batches", str(VAL_BATCHES),
    "--step_scheduler.ckpt_every_steps", "15",
    "--checkpoint.checkpoint_dir", str(CHECKPOINT_DIR),
    # Per-field-position loss: period is one transaction plus its <sep>.
    "--diagnostics.field_loss_period", str(TOKENS_PER_TXN + 1),
]

cfg = apply_overrides(load_config(YAML_CONFIG), OVERRIDES)
if EXP is not None:
    cfg = EXP.apply_training(cfg)   # the experiment's settings win (native YAML types)
    # The run folder is this experiment's own: a re-run of this stage (after a
    # crash) restarts training, so its earlier checkpoints are discarded.
    cfg.setdefault("checkpoint", {})["overwrite"] = True

# Field names and per-field vocabulary sizes come from the tokenizer state
# (plain JSON -- no cuDF import needed here). They label the diagnostic and give
# each field its own random-init baseline, ln(vocab_size of that field).
STATE_PATH = TOKENIZER_DIR / "tokenizer_state.json"
FIELD_NAMES, FIELD_VOCAB = [], {}
if STATE_PATH.exists():
    _state = json.loads(STATE_PATH.read_text())
    # field_names lists every token position (a text field has several);
    # older states only have tokenizer_order, one position per step.
    FIELD_NAMES = list(_state.get("field_names") or _state.get("tokenizer_order") or [])
    FIELD_VOCAB = dict(_state.get("field_vocab") or _state.get("vocab_sizes") or {})
    cfg.setdefault("diagnostics", {})["field_names"] = FIELD_NAMES
    print(f"Fields per transaction: {len(FIELD_NAMES)}  (+1 for <sep>)")
else:
    print(f"NOTE: {STATE_PATH} not found -- the per-field diagnostic will be "
          "reported by index instead of by field name.")

history = train_bank_decoder(cfg, log_every_steps=5)

# The whole history as data (every step, val losses, loss per field), so it can be
# read back later (06_results.ipynb) without parsing the printed output.
history["field_vocab"] = FIELD_VOCAB
(MODEL_DIR / "training_history.json").write_text(json.dumps(history, indent=1, default=float))
print(f"History -> {MODEL_DIR / 'training_history.json'}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(12, 3.6), width_ratios=[2, 1])

ax_loss.plot(history["steps"], history["train_loss"], lw=1.6, label="train")
if history["val_steps"]:
    ax_loss.plot(history["val_steps"], history["val_loss"], "o-", lw=1.6, label="val")
if history.get("objective") in (None, "clm", "mlm"):   # ln(V) is a token-loss baseline
    ax_loss.axhline(history["baseline_loss"], ls="--", c="gray",
                    label=f"ln(V) = {history['baseline_loss']:.2f}")
ax_loss.set_xlabel("optimizer step")
ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title({"mlm": "Masked-token prediction loss",
                   "nppr": "NPPR loss (next transfer + past reconstruction)",
                   "coles": "CoLES contrastive loss"}.get(history.get("objective"),
                                                         "Next-token prediction loss"))
ax_loss.legend(fontsize=8)

ax_lr.plot(history["steps"], history["lr"], lw=1.6, c="tab:orange")
ax_lr.set_xlabel("optimizer step")
ax_lr.set_ylabel("learning rate")
ax_lr.set_title("Warmup + cosine decay")

fig.tight_layout()
plt.show()

# The demo is far too short to converge; what matters here is only that the loss
# moves off ln(V). A flat curve at ~ln(V) means the signal is not reaching the
# model -- check that vocab_size matches the tokenizer and that labels are not
# all -100.
if history["train_loss"]:
    first, last = history["train_loss"][0], history["train_loss"][-1]
    print(f"train loss {first:.3f} -> {last:.3f}   "
          f"({100 * (first - last) / first:+.1f}% vs step 1, "
          f"baseline ln(V) = {history['baseline_loss']:.2f})")
if history["val_loss"]:
    print(f"final val loss {history['val_loss'][-1]:.3f}   "
          f"perplexity {math.exp(min(history['val_loss'][-1], 20)):.1f}")
print(f"checkpoints: {history['checkpoints']}")

### Where did the loss actually drop? Loss by field position

The aggregate curve above hides what is being learned. The corpus is rigidly periodic --
`<bos>` then `[16 field tokens, <sep>]` repeated -- so bucketing the per-token loss by
position within a transaction gives **one number per tokenizer step**.

This matters because the early fall from `ln(V)` is mostly the model learning the
**schema**, not behaviour:

- `<sep>` recurs every `T+1` positions and becomes perfectly predictable almost
  immediately (~6% of all positions).
- Direction, sign and the operational flags have 2-4 values each, so their entropy
  ceiling is ~1-2 nats no matter what.

What decides whether the embeddings carry signal is the **high-cardinality** fields --
counterparty bucket, amount bin, time gap. The right way to read the chart below is
therefore *not* absolute loss but the gap to each field's own random-init baseline,
`ln(vocab_size)` for that step: a bar far short of its baseline is a field the model
genuinely learned; a bar sitting at its baseline is one it has not, however small the
absolute number.

Set `diagnostics.field_loss_period` to `null` in the YAML to skip this.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Per-field vocabulary sizes give each field its own baseline. Re-read them if
# the training cell was skipped this session.
try:
    FIELD_VOCAB
except NameError:
    _sp = TOKENIZER_DIR / "tokenizer_state.json"
    FIELD_VOCAB = (
        dict(json.loads(_sp.read_text()).get("field_vocab")
             or json.loads(_sp.read_text()).get("vocab_sizes") or {}) if _sp.exists() else {}
    )

if not history.get("field_loss"):
    print("No per-field diagnostic recorded.\n"
          "Set --diagnostics.field_loss_period to TOKENS_PER_TXN + 1 and ensure "
          "validation ran at least once (step_scheduler.val_every_steps).")
else:
    field_loss = history["field_loss"]
    period = len(field_loss)
    names = list(history.get("field_names") or [])
    labels_ = [
        names[j] if j < len(names) else ("<sep>/<eos>" if j == period - 1 else f"field {j}")
        for j in range(period)
    ]
    # Each field's own random-init expectation: ln(size of that step's vocabulary).
    # The <sep> slot has no tokenizer step, so it gets no baseline.
    padded_names = names + [None] * (period - len(names))
    baselines = [
        math.log(FIELD_VOCAB[n]) if (n in FIELD_VOCAB and FIELD_VOCAB[n] > 1) else None
        for n in padded_names
    ]

    order = sorted(range(period), key=lambda j: -field_loss[j])
    y = np.arange(period)

    fig, ax = plt.subplots(figsize=(9, 0.42 * period + 1.6))
    ax.barh(y, [field_loss[j] for j in order], color="tab:blue",
            label="validation loss (nats)")
    for row, j in enumerate(order):
        base = baselines[j]
        if base is None:
            continue
        ax.plot([base], [row], marker="|", markersize=18, color="crimson",
                markeredgewidth=2.2, zorder=3)
        ax.text(max(base, field_loss[j]) + 0.12, row,
                f"-{base - field_loss[j]:.2f}", va="center", fontsize=8, color="dimgray")

    ax.set_yticks(y)
    ax.set_yticklabels([labels_[j] for j in order], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("cross-entropy (nats)")
    ax.set_title(f"Validation loss by field position  (step {history['field_loss_step']})")
    ax.plot([], [], marker="|", linestyle="None", markersize=12, color="crimson",
            markeredgewidth=2.2, label="ln(vocab) for that field")
    ax.legend(fontsize=8, loc="lower right")
    ax.margins(x=0.14)
    fig.tight_layout()
    plt.show()

    print(f"{'field':<26}{'loss':>8}{'ln(vocab)':>11}{'learned':>9}{'targets':>12}")
    print("-" * 66)
    for j in order:
        base = baselines[j]
        base_s = f"{base:.2f}" if base is not None else "-"
        gain_s = f"{base - field_loss[j]:.2f}" if base is not None else "-"
        print(f"{labels_[j]:<26}{field_loss[j]:>8.3f}{base_s:>11}{gain_s:>9}"
              f"{int(history['field_loss_counts'][j]):>12,}")

    scored = sorted(
        ((baselines[j] - field_loss[j], labels_[j]) for j in range(period)
         if baselines[j] is not None),
        reverse=True,
    )
    if scored:
        print(f"\nMost learned : {', '.join(n for _, n in scored[:3])}")
        print(f"Least learned: {', '.join(n for _, n in scored[-3:])}")
        print("\nA field near its ln(vocab) baseline is one the model has not learned "
              "yet -- for the high-cardinality fields (counterparty, amount, time gap) "
              "that is the signal to train longer.")

### Full distributed run

Scaling to multiple GPUs is a `torchrun` wrapper -- no code change.
`scripts/train_bank_decoder.py` is the CLI entry point for the same loop; when
`WORLD_SIZE > 1` it initialises NCCL, wraps the model in `DistributedDataParallel` and
switches to a `DistributedSampler`. Gradient accumulation adjusts automatically, so
`global_batch_size` means the same thing on 1 GPU and on 8.

DDP rather than FSDP2: at ~29M parameters the model, gradients and optimizer state fit
on one device many times over, so sharding them would only add communication.

In [ ]:
print("For a full distributed run (8 GPUs):\n")
print(f"  torchrun --nproc-per-node=8 {TRAIN_SCRIPT} \\")
print(f"      -c {YAML_CONFIG} \\")
print(f"      --dataset.data_path {TRAIN_CORPUS} \\")
print(f"      --dataset.vocab_path {VOCAB_PATH} \\")
print(f"      --validation_dataset.data_path {VAL_CORPUS} \\")
print(f"      --validation_dataset.vocab_path {VOCAB_PATH} \\")
print(f"      --model.config.vocab_size {VOCAB_SIZE} \\")
print(f"      --step_scheduler.max_steps 3000 \\")
print(f"      --step_scheduler.global_batch_size 128 \\")
print(f"      --step_scheduler.local_batch_size 16 \\")
print(f"      --checkpoint.checkpoint_dir {CHECKPOINT_DIR}")
print("\nSame loop on one GPU, if you prefer a script to this notebook:\n")
print(f"  python {TRAIN_SCRIPT} -c {YAML_CONFIG} \\")
print(f"      --dataset.data_path {TRAIN_CORPUS} \\")
print(f"      --dataset.vocab_path {VOCAB_PATH} \\")
print(f"      --model.config.vocab_size {VOCAB_SIZE} \\")
print(f"      --step_scheduler.max_steps 3000 \\")
print("      --history-out models/bank-decoder/loss_history.json")

## 2. Locate the Checkpoint for Notebook 04

Checkpoints are written by `save_pretrained()` in HuggingFace format (`config.json` +
safetensors), which is what `save_consolidated: true` produced under NeMo and what
`AutoModelForCausalLM.from_pretrained()` expects -- so notebook 04 needs no change.

Two details this cell relies on:

- **It prefers the best-validation checkpoint, not the last one.** Once val loss
  flattens, the final step is a worse model than an earlier one. `save_best: true`
  writes a checkpoint on every validation improvement, and neither the best nor the
  final step is ever pruned by `keep_last_n`.
- **Completeness is judged by `training_state.json`, not `config.json`.**
  `save_pretrained` writes the config *before* the weights, so a save killed
  mid-serialization -- a full disk, most likely -- leaves a directory that looks
  loadable but holds a truncated safetensors file. The marker is written last, so
  partial checkpoints are invisible to discovery instead of being preferred by mtime.

If `history` is not in scope (a script or `torchrun` run), it falls back to the newest
complete checkpoint on disk.

In [ ]:
from src.train_decoder import CHECKPOINT_MARKER, completed_checkpoints

latest = None
source = None

# Prefer what the run itself reported -- best-validation over final, since the
# last step is not the best model once val loss flattens. Fall back to a
# filesystem scan so this cell also works after a script/torchrun run.
if "history" in dir():
    if history.get("best_checkpoint"):
        latest, source = Path(history["best_checkpoint"]), "best validation"
    elif history.get("final_checkpoint"):
        latest, source = Path(history["final_checkpoint"]), "final step"

if latest is None:
    # Keyed on training_state.json, not config.json: save_pretrained writes the
    # config before the weights, so a save killed mid-write (full disk) leaves a
    # directory that looks loadable but is truncated. The marker is written last.
    completed = completed_checkpoints(CHECKPOINT_DIR)
    if completed:
        latest = completed[max(completed)]
        source = f"latest complete on disk (step {max(completed)})"

if latest is not None:
    record = {
        "checkpoint": str(latest),
        "selected_by": source,
        "vocab_size": VOCAB_SIZE,
        "seq_length": SEQ_LENGTH,
        "tokens_per_transaction": TOKENS_PER_TXN,
        "corpus_report": str(REPORT_PATH),
        # Which tokenizer the model was trained under (NB04 checks it).
        "tokenizer_fingerprint": report.get("tokenizer_fingerprint"),
    }
    if "history" in dir():
        record["best_step"] = history.get("best_step")
        record["best_val_loss"] = history.get("best_val_loss")
        record["objective"] = history.get("objective", "clm")
    (MODEL_DIR / "latest_checkpoint.json").write_text(json.dumps(record, indent=2))

    print(f"Checkpoint for NB04: {latest}")
    print(f"  selected by: {source}")
    print("  files:", sorted(p.name for p in latest.iterdir())[:8])
    if "history" in dir() and history.get("best_val_loss") is not None:
        print(f"  best val loss {history['best_val_loss']:.4f} "
              f"at step {history['best_step']}")
    print(f"\nRecorded -> {MODEL_DIR / 'latest_checkpoint.json'}")
else:
    partial = [
        p for p in CHECKPOINT_DIR.glob("step_*")
        if p.is_dir() and not (p / CHECKPOINT_MARKER).exists()
    ]
    print(f"No complete checkpoint under {CHECKPOINT_DIR}.")
    if partial:
        print(f"  {len(partial)} partial (unfinished) checkpoint(s) found and "
              f"ignored: {[p.name for p in partial]}")
        print("  These are truncated writes -- delete them and check free disk space.")
    print("  Re-run the training cell (or a longer run) before notebook 04.")

## Summary

The model is pretrained on account sequences with next-token prediction and no labels.
The demo run only shows the loss moving; run a few thousand steps before expecting the
embeddings to carry signal in notebook 05.

**Outputs**

| File | Used by |
|---|---|
| `models/bank-decoder/checkpoints/...` | NB04 (inference) |
| `models/bank-decoder/latest_checkpoint.json` | NB04 (checkpoint discovery) |

Continue to [04_inference_embedding_extraction.ipynb](./04_inference_embedding_extraction.ipynb).